In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
print(os.listdir('/content/drive/MyDrive/'))


['Colab Notebooks', 'SCTEC', 'IMG-20260213-WA0004.jpeg', 'analise da academia.gsheet', 'LOJAS.gsheet', 'Untitled0.ipynb', 'Untitled1.ipynb', 'Larápios', 'Salvo do Chrome', 'vinheta_de_larapios_17x0_6cm_600dpi.png', 'Varejo (3).csv', '.ipynb_checkpoints', 'Varejo.csv', 'Miniprojeto_AishaRodriguesDaSilva_QA-ADPY__2026-1-4']


In [5]:
import pandas as pd

In [6]:
Varejo = pd.read_csv("/content/drive/MyDrive/Varejo.csv", sep=";")

Varejo = Varejo.loc[:, ~Varejo.columns.str.contains('^Unnamed')]

In [7]:
print(Varejo.columns)

Index(['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG',
       'PR_ID', 'PR_CAT', 'PR_NOME'],
      dtype='object')


In [8]:
print(Varejo.shape)
print(Varejo.dtypes)

(830000, 10)
DATA         object
CO_ID         int64
CL_ID         int64
CL_GENERO    object
CL_EC         int64
CL_FHL        int64
CL_SEG       object
PR_ID         int64
PR_CAT       object
PR_NOME      object
dtype: object


In [9]:
# Verificando valores nulos
print("\nValores nulos:")
print(Varejo.isnull().sum())

# Verificando duplicatas
print("\nDuplicados:")
print(Varejo.duplicated().sum())

# Verificando categorias ausentes
print("\nCategorias ausentes:")
print((Varejo == "#N/D").sum())

# Substituindo categorias ausentes por "Sem Categoria"
if "#N/D" in Varejo["PR_CAT"].values:
    Varejo["PR_CAT"] = Varejo["PR_CAT"].replace("#N/D", "Sem Categoria")

# Verificando se ainda existem categorias "#N/D"
print("\n#N/D restantes:")
print((Varejo["PR_CAT"] == "#N/D").sum())


Valores nulos:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

Duplicados:
96553

Categorias ausentes:
DATA            0
CO_ID           0
CL_ID           0
CL_GENERO       0
CL_EC           0
CL_FHL          0
CL_SEG          0
PR_ID           0
PR_CAT       3650
PR_NOME      3650
dtype: int64

#N/D restantes:
0


In [10]:
# Eliminando linhas completamente duplicadas porque representam registros repetidos na base
Varejo = Varejo.drop_duplicates().copy()

# fazendo a coluna DATA ir para o formato datetime
Varejo["DATA"] = pd.to_datetime(Varejo["DATA"], format="%d/%m/%Y")

# fazendo a coluna CL_FHL ir para interger
Varejo["CL_FHL"] = Varejo["CL_FHL"].astype(int)

# Criando o DataFrame final com os dados limpos
df_limpo = Varejo.copy()

df_limpo.to_csv("df_limpo.csv", index=False, sep=";")

# os dados CO_ID, CL_ID e PR_ID não serão convertidos pois são códigos indentificadores, portanto strings, apesar de para nós parecer um numero. O dado CL_EC também não, pois trata-se de uma lista. Não foi necessário transformar nenhum dado em float neste caso.


In [11]:
# Agrupando os itens pelo ID da compra
itens_por_compra = Varejo.groupby("CO_ID").size()

print(itens_por_compra.head())

CO_ID
1000    46
1040    15
1078    63
1082    62
1103     6
dtype: int64


In [12]:
#valor dos filhos do cliente
media = Varejo['CL_FHL'].mean()

print("\n Media ")
print(media)

mediana = Varejo['CL_FHL'].median()

print("\n Mediana")
print(mediana)

desvio_padrao = Varejo['CL_FHL'].std()

print("\n desvio padrão")
print(desvio_padrao)

moda = Varejo['CL_FHL'].mode()

print("\n moda")
print(moda)

maximo = Varejo['CL_FHL'].max()

print("\n máximo de filhos")
print(maximo)

minimo = Varejo['CL_FHL'].min()

print("\n mínimo")
print(minimo)

total = Varejo['CL_FHL'].count()

print("\n Contagem")
print(total)

Quartis1 = Varejo['CL_FHL'].quantile(0.25)
Quartis2 = Varejo['CL_FHL'].quantile(0.50)
Quartis3 = Varejo['CL_FHL'].quantile(0.75)

print("\n QUARTIS")
print("25%:")
print(Quartis1)
print("50%:")
print(Quartis2)
print("75%:")
print(Quartis3)


 Media 
1.1460487260838206

 Mediana
0.0

 desvio padrão
1.4169173697854265

 moda
0    0
Name: CL_FHL, dtype: int64

 máximo de filhos
4

 mínimo
0

 Contagem
733447

 QUARTIS
25%:
0.0
50%:
0.0
75%:
2.0


In [13]:
#padrões de agrupamento por gênero e compras por genero

# Agrupando os itens por gênero do cliente
genero_do_cliente = Varejo.groupby("CL_GENERO").size()

print("\n genero do cliente")
print(genero_do_cliente)

# Contando compras diferentes por gênero
compras_por_genero = Varejo.groupby("CL_GENERO")["CO_ID"].nunique()

print("\n compra por genero")
print(compras_por_genero)


 genero do cliente
CL_GENERO
F    382427
M    351020
dtype: int64

 compra por genero
CL_GENERO
F    9615
M    8856
Name: CO_ID, dtype: int64


Então nossas Conclusões são:

- A análise da coluna CL_FHL ajudou a entender melhor a quantidade de filhos dos clientes, usando cálculos como média, mediana, moda, mínimo, máximo e quartis.
- O agrupamento por CL_GENERO mostrou diferenças na quantidade de itens registrados entre os gêneros.
- Também foi possível analisar a quantidade de compras diferentes para cada gênero usando CL_GENERO e CO_ID.
- Foi possível perceber que o CO_ID pode aparecer várias vezes, porque cada linha representa um item de uma compra e não uma compra inteira.
- Na limpeza dos dados, foram removidas linhas duplicadas e a coluna DATA foi transformada para a forma de data.
- Os valores #N/D encontrados nas categorias foram substituídos por Sem Categoria.